# Auditing Racial Bias in Healthcare Utilization Prediction
## A Fairness and Explainability Audit Aligned with SDAIA AI Ethics Principles

**Course:** CSC 594 – Ethical Issues in Artificial Intelligence  
**Institution:** King Saud University, College of Computer and Information Sciences  
**Domain:** Healthcare

---

### Objective

This project performs a **Technical AI Ethics Audit** on a healthcare utilization prediction model using the Medical Expenditure Panel Survey (MEPS) dataset. We act as an AI "Ethics Auditor" to:

1. **Detect bias** in a model that predicts high healthcare utilization (≥10 visits/year)
2. **Explain** the model's decision-making process using interpretability techniques
3. **Mitigate** discovered bias using algorithmic fairness interventions
4. **Quantify** the Fairness–Accuracy tradeoff

### Ethical Hypothesis

> *We hypothesize that a healthcare utilization prediction model exhibits racial bias — systematically under-predicting healthcare needs for non-White patients — because historical utilization data reflects unequal access to care, not actual medical need.*

This replicates findings from the landmark **Obermeyer et al. (2019)** study published in *Science*, which exposed racial bias in a widely-used healthcare algorithm affecting millions of patients.

### Alignment with SDAIA AI Ethics Principles

| SDAIA Principle | How This Audit Addresses It |
|---|---|
| **Fairness** | We measure Demographic Parity and Equalized Odds across racial groups |
| **Transparency & Explainability** | We use SHAP to explain which features drive predictions |
| **Reliability & Safety** *(Safety dimension only)* | We reduce false-negative miss rates for vulnerable groups via mitigation. P5's Reliability dimension — system-as-designed behaviour — is out of scope. |
| **Accountability & Responsibility** *(human-oversight requirement only)* | Individual SHAP explanations enable case-level human oversight. Broader P7 obligations — governance, liability, lifecycle management — are out of scope. |

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score,
                             classification_report, confusion_matrix)

from fairlearn.metrics import (MetricFrame, demographic_parity_difference,
                                demographic_parity_ratio, equalized_odds_difference,
                                false_positive_rate, false_negative_rate,
                                selection_rate, count)
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds

import shap

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('colorblind')

print("All libraries loaded successfully.")
print(f"Fairlearn version: {__import__('fairlearn').__version__}")
print(f"SHAP version: {shap.__version__}")

## 2. Data Loading & Exploration

The **Medical Expenditure Panel Survey (MEPS)** is a large-scale survey conducted by the U.S. Agency for Healthcare Research and Quality (AHRQ). It collects data on healthcare services used, costs, frequency of services, and demographics.

We use **Panel 19 (2015)** and follow the preprocessing from IBM AIF360's benchmark, which:
- Creates a binary **`Race`** feature (MEPS code `RACE`): `White` (non-Hispanic White) vs. `Non-White` (all others)
- Computes **`Utilization`** (MEPS code `UTILIZATION`) = total visits across outpatient, ER, inpatient, and home health
- Binarizes utilization: `1` if ≥10 visits (high utilizer), `0` otherwise

In [ ]:
# Load raw MEPS data (Panel 19, 2015)
filepath = 'venv/lib/python3.12/site-packages/aif360/data/raw/meps/h181.csv'
df_raw = pd.read_csv(filepath)
print(f"Raw dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns")

# --- Preprocessing (following AIF360 benchmark) ---
# Note: this whole block uses MEPS-canonical column codes — we follow the
# published AIF360 recipe so our row counts and metrics are comparable to the
# fairness literature. We rename to human-readable column names ONCE at the end
# of this cell, after df_model is built; everything downstream uses those.

# 1. Create RACE: White (non-Hispanic White) vs Non-White
df_raw['RACE'] = df_raw.apply(
    lambda row: 1 if (row['HISPANX'] == 2 and row['RACEV2X'] == 1) else 0, axis=1
)

# 2. Filter to Panel 19
df = df_raw[df_raw['PANEL'] == 19].copy()
print(f"After filtering to Panel 19: {len(df):,} rows")

# 3. Rename panel/round-specific columns to AIF360-canonical codes
df = df.rename(columns={
    'FTSTU53X': 'FTSTU', 'ACTDTY53': 'ACTDTY', 'HONRDC53': 'HONRDC',
    'RTHLTH53': 'RTHLTH', 'MNHLTH53': 'MNHLTH', 'CHBRON53': 'CHBRON',
    'JTPAIN53': 'JTPAIN', 'PREGNT53': 'PREGNT', 'WLKLIM53': 'WLKLIM',
    'ACTLIM53': 'ACTLIM', 'SOCLIM53': 'SOCLIM', 'COGLIM53': 'COGLIM',
    'EMPST53': 'EMPST', 'REGION53': 'REGION', 'MARRY53X': 'MARRY',
    'AGE53X': 'AGE', 'POVCAT15': 'POVCAT', 'INSCOV15': 'INSCOV',
})

# 4. Remove rows with invalid/missing values
df = df[df['REGION'] >= 0]
df = df[df['AGE'] >= 0]
df = df[df['MARRY'] >= 0]
df = df[df['ASTHDX'] >= 0]

cat_cols_meps = [
    'FTSTU','ACTDTY','HONRDC','RTHLTH','MNHLTH','HIBPDX','CHDDX','ANGIDX',
    'EDUCYR','HIDEG','MIDX','OHRTDX','STRKDX','EMPHDX','CHBRON','CHOLDX',
    'CANCERDX','DIABDX','JTPAIN','ARTHDX','ARTHTYPE','ASTHDX','ADHDADDX',
    'PREGNT','WLKLIM','ACTLIM','SOCLIM','COGLIM','DFHEAR42','DFSEE42',
    'ADSMOK42','PHQ242','EMPST','POVCAT','INSCOV',
]
df = df[(df[cat_cols_meps] >= -1).all(axis=1)]
print(f"After removing invalid values: {len(df):,} rows")

# 5. Compute UTILIZATION (total visits across all care types)
df['UTILIZATION'] = (df['OBTOTV15'] + df['OPTOTV15'] +
                     df['ERTOT15'] + df['IPNGTD15'] + df['HHTOTD15'])
df['UTILIZATION'] = (df['UTILIZATION'] >= 10).astype(int)

# 6. Select features (still using MEPS codes)
feature_cols_meps = [
    'REGION','AGE','SEX','RACE','MARRY','FTSTU','ACTDTY','HONRDC',
    'RTHLTH','MNHLTH','HIBPDX','CHDDX','ANGIDX','MIDX','OHRTDX',
    'STRKDX','EMPHDX','CHBRON','CHOLDX','CANCERDX','DIABDX','JTPAIN',
    'ARTHDX','ARTHTYPE','ASTHDX','ADHDADDX','PREGNT','WLKLIM','ACTLIM',
    'SOCLIM','COGLIM','DFHEAR42','DFSEE42','ADSMOK42','PCS42','MCS42',
    'K6SUM42','PHQ242','EMPST','POVCAT','INSCOV',
]
target_col_meps = 'UTILIZATION'

df_model = df[feature_cols_meps + [target_col_meps]].copy()

# 7. Rename columns to human-readable labels.
#    The MEPS codebook codes are kept in the schema legend below the sample
#    cell for traceability. Everything downstream of this cell uses these
#    readable names.
RENAMES = {
    'REGION':      'Census region',
    'AGE':         'Age',
    'SEX':         'Sex',
    'RACE':        'Race',
    'MARRY':       'Marital status',
    'FTSTU':       'Full-time student',
    'ACTDTY':      'Active military',
    'HONRDC':      'Honorable discharge',
    'RTHLTH':      'Self-rated health',
    'MNHLTH':      'Self-rated mental health',
    'HIBPDX':      'High BP dx',
    'CHDDX':       'Coronary heart disease dx',
    'ANGIDX':      'Angina dx',
    'MIDX':        'Heart attack dx',
    'OHRTDX':      'Other heart disease dx',
    'STRKDX':      'Stroke dx',
    'EMPHDX':      'Emphysema dx',
    'CHBRON':      'Chronic bronchitis dx',
    'CHOLDX':      'High cholesterol dx',
    'CANCERDX':    'Cancer dx',
    'DIABDX':      'Diabetes dx',
    'JTPAIN':      'Joint pain',
    'ARTHDX':      'Arthritis dx',
    'ARTHTYPE':    'Arthritis type',
    'ASTHDX':      'Asthma dx',
    'ADHDADDX':    'ADHD/ADD dx',
    'PREGNT':      'Currently pregnant',
    'WLKLIM':      'Walking limitation',
    'ACTLIM':      'Activity limitation',
    'SOCLIM':      'Social limitation',
    'COGLIM':      'Cognitive limitation',
    'DFHEAR42':    'Hearing difficulty',
    'DFSEE42':     'Vision difficulty',
    'ADSMOK42':    'Smoking status',
    'PCS42':       'SF-12 Physical Composite',
    'MCS42':       'SF-12 Mental Composite',
    'K6SUM42':     'K6 distress score',
    'PHQ242':      'PHQ-2 depression',
    'EMPST':       'Employment status',
    'POVCAT':      'Income category',
    'INSCOV':      'Insurance coverage',
    'UTILIZATION': 'Utilization',
}
df_model = df_model.rename(columns=RENAMES)
feature_cols = [RENAMES[c] for c in feature_cols_meps]
target_col   = RENAMES[target_col_meps]

print(f"\nFinal dataset: {df_model.shape[0]:,} rows x {df_model.shape[1]:,} columns")
print(f"Features: {len(feature_cols)}")
print(f"\nTarget distribution:")
print(df_model[target_col].value_counts().rename({0: '< 10 visits', 1: '>= 10 visits'}))
print(f"\nRace distribution:")
print(df_model['Race'].value_counts().rename({1: 'White', 0: 'Non-White'}))


### Sample of the Cleaned Data

Six rows that span the four (RACE × UTILIZATION) cells of the dataset, to give a sense of the feature scales and MEPS coding conventions before we move to modelling.


In [ ]:
# Six rows spanning the four (Race x Utilization) combinations.
sample_cols = ['Age', 'Sex', 'Race', 'Income category', 'Insurance coverage',
               'Self-rated health', 'High BP dx', 'Utilization']
_parts = []
for _rc in [1, 0]:                    # 1 = White, 0 = Non-White
    for _ut in [0, 1]:                # 0 = low utilisation, 1 = high
        _n = 2 if _rc == _ut else 1   # asymmetric so we get exactly 6 rows
        _parts.append(
            df_model[(df_model['Race'] == _rc) & (df_model['Utilization'] == _ut)]
            [sample_cols].head(_n)
        )
sample = pd.concat(_parts).reset_index(drop=True)
print('Sample of the cleaned dataset (6 rows spanning Race x Utilization):')
print(sample.to_string())
print()
print('Schema (readable label  <-  MEPS codebook code  =  meaning):')
print('  Sex                   <-  SEX           1 = Male, 2 = Female')
print('  Race                  <-  RACE          1 = White (non-Hispanic), 0 = Non-White')
print('  Income category       <-  POVCAT        1 = Poor, 2 = Near poor, 3 = Low, 4 = Middle, 5 = High')
print('  Insurance coverage    <-  INSCOV        1 = Any private, 2 = Public only, 3 = Uninsured')
print('  Self-rated health     <-  RTHLTH        1 = Excellent ... 5 = Poor')
print('  High BP dx            <-  HIBPDX        1 = Yes, 2 = No, -1 = Inapplicable')
print('  Utilization           <-  UTILIZATION   Target: 1 if (OBTOTV15+OPTOTV15+ERTOT15+IPNGTD15+HHTOTD15) >= 10')


In [ ]:
# Exploratory visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Target distribution
df_model['Utilization'].value_counts().plot(kind='bar', ax=axes[0], color=['#2196F3', '#FF5722'])
axes[0].set_title('Target: Healthcare Utilization')
axes[0].set_xticklabels(['< 10 visits', '>= 10 visits'], rotation=0)
axes[0].set_ylabel('Count')

# 2. Race distribution
race_counts = df_model['Race'].value_counts()
race_counts.plot(kind='bar', ax=axes[1], color=['#FF9800', '#4CAF50'])
axes[1].set_title('Race Distribution')
axes[1].set_xticklabels(['Non-White', 'White'], rotation=0)
axes[1].set_ylabel('Count')

# 3. Utilization rate BY race (key disparity visualization)
util_by_race = df_model.groupby('Race')['Utilization'].mean() * 100
util_by_race.plot(kind='bar', ax=axes[2], color=['#FF9800', '#4CAF50'])
axes[2].set_title('High Utilization Rate by Race (%)')
axes[2].set_xticklabels(['Non-White', 'White'], rotation=0)
axes[2].set_ylabel('% with >= 10 visits')
for i, v in enumerate(util_by_race):
    axes[2].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/fig1_data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey observation: The utilization rates across racial groups reveal the baseline disparity")
print("that our model may learn and amplify.")


## 3. Model Training

We train a single classifier — a **Gradient Boosting Classifier** (max_depth = 4, n_estimators = 100, learning_rate = 0.1) — to predict high healthcare utilization. Gradient Boosting is a non-linear ensemble; its "black-box" character makes it a natural target for the SHAP explainability work in §5.

We evaluate the model on overall accuracy and on **per-group performance** (White vs. Non-White) to identify initial signs of disparate impact.

In [ ]:
# Prepare features and target
X = df_model[feature_cols].values
y = df_model[target_col].values
sensitive_features = df_model['Race'].values  # 1=White, 0=Non-White

# Train/test split (stratified by target)
X_train, X_test, y_train, y_test, sf_train, sf_test = train_test_split(
    X, y, sensitive_features, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set:     {X_test.shape[0]:,} samples")
print(f"\nTraining set race distribution: White={sum(sf_train==1)}, Non-White={sum(sf_train==0)}")
print(f"Test set race distribution:     White={sum(sf_test==1)}, Non-White={sum(sf_test==0)}")

In [ ]:
# Train the Gradient Boosting Classifier (no scaling needed for tree ensembles).
gb_model = GradientBoostingClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1, random_state=RANDOM_STATE
)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)
y_proba_gb = gb_model.predict_proba(X_test)[:, 1]

# Overall performance
print("=" * 60)
print("OVERALL MODEL PERFORMANCE — Gradient Boosting")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_gb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_gb, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_gb, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_gb, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba_gb):.4f}")

In [ ]:
# Per-group performance breakdown (early sign of bias)
print("=" * 60)
print("PER-GROUP PERFORMANCE — Gradient Boosting (White vs Non-White)")
print("=" * 60)

race_labels = {0: 'Non-White', 1: 'White'}
for race_val, race_name in race_labels.items():
    mask = sf_test == race_val
    if mask.sum() == 0:
        continue
    acc  = accuracy_score (y_test[mask], y_pred_gb[mask])
    prec = precision_score(y_test[mask], y_pred_gb[mask], zero_division=0)
    rec  = recall_score   (y_test[mask], y_pred_gb[mask], zero_division=0)
    f1   = f1_score       (y_test[mask], y_pred_gb[mask], zero_division=0)
    print(f"\n  {race_name} (n={mask.sum()}):")
    print(f"    Accuracy={acc:.4f}  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}")

## 4. Fairness Audit (Phase 2)

We now formally audit the models for bias using **Microsoft Fairlearn**'s `MetricFrame`. We compute the following fairness metrics:

| Metric | Definition | Fairness Threshold |
|---|---|---|
| **Demographic Parity Difference** | Difference in positive prediction rates between groups | 0 = perfectly fair |
| **Demographic Parity Ratio** | Ratio of positive prediction rates | 1.0 = perfectly fair, <0.8 = potential violation |
| **Equalized Odds Difference** | Max difference in TPR and FPR between groups | 0 = perfectly fair |
| **False Negative Rate** | Rate at which the model misses positive cases (per group) | Equal across groups = fair |

These metrics directly operationalize the **SDAIA Fairness & Non-Discrimination** principle.

In [ ]:
# We focus the fairness audit on the Gradient Boosting model (higher accuracy, more likely to be deployed)
y_pred = y_pred_gb  # Use GB predictions for the main audit

# Compute fairness metrics using Fairlearn MetricFrame
sf_test_labeled = pd.Series(sf_test).map({1: 'White', 0: 'Non-White'}).values

metric_frame = MetricFrame(
    metrics={
        'Accuracy': accuracy_score,
        'Precision': lambda y_t, y_p: precision_score(y_t, y_p, zero_division=0),
        'Recall': lambda y_t, y_p: recall_score(y_t, y_p, zero_division=0),
        'F1-Score': lambda y_t, y_p: f1_score(y_t, y_p, zero_division=0),
        'Selection Rate': selection_rate,
        'False Positive Rate': false_positive_rate,
        'False Negative Rate': false_negative_rate,
        'Count': count,
    },
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sf_test_labeled
)

print("=" * 70)
print("FAIRNESS AUDIT: Per-Group Metrics (Gradient Boosting)")
print("=" * 70)
print("\nMetrics by Race Group:")
print(metric_frame.by_group.to_string())

print("\n" + "=" * 70)
print("FAIRNESS AUDIT: Overall Fairness Metrics")
print("=" * 70)
print(f"\nDemographic Parity Difference: {demographic_parity_difference(y_test, y_pred, sensitive_features=sf_test_labeled):.4f}")
print(f"Demographic Parity Ratio:      {demographic_parity_ratio(y_test, y_pred, sensitive_features=sf_test_labeled):.4f}")
print(f"Equalized Odds Difference:     {equalized_odds_difference(y_test, y_pred, sensitive_features=sf_test_labeled):.4f}")

dp_ratio = demographic_parity_ratio(y_test, y_pred, sensitive_features=sf_test_labeled)
if dp_ratio < 0.8:
    print(f"\n⚠ WARNING: Demographic Parity Ratio ({dp_ratio:.4f}) is below the 0.8 threshold (4/5ths rule).")
    print("  This indicates a potential disparate impact under anti-discrimination standards.")

### Quantifying the Amplification

A fairness gap can arise either because the model *reflects* an underlying
disparity in the data or because it *amplifies* one. To distinguish these,
we compare each group's ground-truth positive rate (how often patients are
actually high-utilizers) against the model's selection rate (how often the
model predicts them as high-utilizers).


In [ ]:
# Base-rate analysis: is the model just reflecting reality, or amplifying it?
# Compare ground-truth positive rates per group to the model's selection rates.
import numpy as np

base_rate_white    = (y_test[sf_test == 1] == 1).mean()
base_rate_nonwhite = (y_test[sf_test == 0] == 1).mean()
sel_rate_white    = (y_pred_gb[sf_test == 1] == 1).mean()
sel_rate_nonwhite = (y_pred_gb[sf_test == 0] == 1).mean()

base_ratio  = base_rate_white / base_rate_nonwhite
model_ratio = sel_rate_white  / sel_rate_nonwhite
amplification = model_ratio / base_ratio

print("=" * 70)
print("BASE-RATE vs MODEL SELECTION RATE: Does the model amplify disparity?")
print("=" * 70)
print(f"\nGround-truth high-utilization rate (P[Y=1 | RACE]):")
print(f"  White:      {base_rate_white*100:5.2f}%")
print(f"  Non-White:  {base_rate_nonwhite*100:5.2f}%")
print(f"  Ratio (W/NW): {base_ratio:.3f}x")

print(f"\nModel selection rate (P[Ŷ=1 | RACE]):")
print(f"  White:      {sel_rate_white*100:5.2f}%")
print(f"  Non-White:  {sel_rate_nonwhite*100:5.2f}%")
print(f"  Ratio (W/NW): {model_ratio:.3f}x")

print(f"\nAmplification factor: {amplification:.3f}x")
print(f"  ({(amplification - 1) * 100:+.1f}% beyond the underlying base-rate disparity)")
print(f"\nInterpretation: a value > 1 means the model widens the disparity")
print(f"present in the training data. This is the empirical version of the")
print(f"Obermeyer et al. (2019) finding for our model.")


In [ ]:
# Visualize fairness metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Selection Rate by group (Demographic Parity)
sr_by_group = metric_frame.by_group['Selection Rate']
sr_by_group.plot(kind='bar', ax=axes[0], color=['#FF9800', '#4CAF50'])
axes[0].set_title('Selection Rate by Race\n(Demographic Parity)')
axes[0].set_ylabel('P(Predicted Positive)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
for i, v in enumerate(sr_by_group):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

# 2. False Negative Rate by group
fnr_by_group = metric_frame.by_group['False Negative Rate']
fnr_by_group.plot(kind='bar', ax=axes[1], color=['#FF9800', '#4CAF50'])
axes[1].set_title('False Negative Rate by Race\n(Missed High-Utilizers)')
axes[1].set_ylabel('FNR')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
for i, v in enumerate(fnr_by_group):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

# 3. Recall by group (ability to identify high utilizers)
rec_by_group = metric_frame.by_group['Recall']
rec_by_group.plot(kind='bar', ax=axes[2], color=['#FF9800', '#4CAF50'])
axes[2].set_title('Recall by Race\n(Ability to Identify High Utilizers)')
axes[2].set_ylabel('Recall (TPR)')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)
for i, v in enumerate(rec_by_group):
    axes[2].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/fig2_fairness_audit.png', dpi=150, bbox_inches='tight')
plt.show()

### Fairness Audit Findings

The metrics above reveal measurable disparities in how the model treats White vs. Non-White patients:

- **Selection Rate Gap**: The model predicts "high utilization" at different rates for the two groups, violating Demographic Parity.
- **False Negative Rate Disparity**: A higher FNR for one group means the model is more likely to *miss* patients who actually need intensive care — a direct patient safety concern.
- **Recall Gap**: Unequal recall means the model's ability to identify high-utilizers depends on the patient's race.

**SDAIA Alignment**: These findings demonstrate a violation of the **Fairness** principle (which requires eliminating bias, discrimination, or stigmatization in AI systems). The model's predictions are not race-neutral, potentially leading to unequal healthcare resource allocation.

## 5. Explainability Audit — SHAP Analysis

We use **SHAP (SHapley Additive exPlanations)** to explain the Gradient Boosting model's predictions. SHAP values decompose each prediction into the contribution of each feature, answering:

- **Which features matter most** for the model's decisions? (Global interpretability)
- **Does the `Race` feature directly influence predictions?** (Direct bias)
- **Do proxy features (e.g., `Income category`, `Insurance coverage`) carry indirect racial bias?** (Proxy discrimination)

This directly operationalizes the **SDAIA Transparency & Explainability** principle.

In [ ]:
# Compute SHAP values for Gradient Boosting model
explainer = shap.TreeExplainer(gb_model)
shap_values = explainer.shap_values(X_test)
assert shap_values.shape == X_test.shape, \
    f"Unexpected SHAP shape {shap_values.shape}; expected {X_test.shape}"

# X_test is a numpy array (no column names); attach the readable feature_cols
# names (defined in cell 4 after the column rename) for SHAP plots.
X_test_df = pd.DataFrame(X_test, columns=feature_cols)

print(f"SHAP values computed for {X_test.shape[0]} test samples across {X_test.shape[1]} features.")


In [ ]:
# 1. SHAP Summary Plot — Global Feature Importance
print("SHAP Summary Plot: Which features drive the model's predictions?")
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_test_df, show=False, max_display=20)
plt.title("SHAP Summary Plot — Feature Impact on High Utilization Prediction", fontsize=14)
plt.tight_layout()
plt.savefig('figures/fig3_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2. SHAP Dependence Plots — Direct and Proxy Bias Investigation
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Direct effect of Race
shap.dependence_plot('Race', shap_values, X_test_df, interaction_index=None,
                     ax=axes[0], show=False)
axes[0].set_title('Direct Effect of Race\non Prediction')

# Proxy: Income category — correlated with Race
shap.dependence_plot('Income category', shap_values, X_test_df, interaction_index='Race',
                     ax=axes[1], show=False)
axes[1].set_title('Income category\ncolored by Race')

# Proxy: Insurance coverage — correlated with Race
shap.dependence_plot('Insurance coverage', shap_values, X_test_df, interaction_index='Race',
                     ax=axes[2], show=False)
axes[2].set_title('Insurance coverage\ncolored by Race')

plt.suptitle('Investigating Direct and Proxy Racial Bias via SHAP', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/fig4_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nProxy bias analysis:")
print("- If Income category / Insurance coverage show different SHAP patterns when colored by Race,")
print("  it confirms these features act as racial proxies — encoding race indirectly.")
print("- This means removing the Race feature alone would NOT eliminate racial bias.")


In [ ]:
# 3. SHAP Force Plots — Individual Prediction Explanations
# Compare explanations for a White patient and a Non-White patient

# Find a White patient predicted as high-utilizer
white_high_idx = np.where((sf_test == 1) & (y_pred_gb == 1) & (y_test == 1))[0]
# Find a Non-White patient predicted as low-utilizer (missed case)
nonwhite_missed_idx = np.where((sf_test == 0) & (y_test == 1) & (y_pred_gb == 0))[0]

print("Individual Prediction Explanations:")
print("=" * 60)

if len(white_high_idx) > 0:
    idx = white_high_idx[0]
    assert y_test[idx] == 1 and y_pred_gb[idx] == 1, \
        "Case 1 must be a true positive"
    print(f"\nCase 1: White patient correctly identified as high-utilizer")
    print(f"  True label: {y_test[idx]}, Predicted: {y_pred_gb[idx]}")
    shap.force_plot(explainer.expected_value, shap_values[idx], X_test_df.iloc[idx],
                    matplotlib=True, show=False)
    plt.title("White Patient — Correctly Identified as High Utilizer", fontsize=12)
    plt.tight_layout()
    plt.savefig('figures/fig5a_shap_force_white.png', dpi=150, bbox_inches='tight')
    plt.show()

if len(nonwhite_missed_idx) > 0:
    idx = nonwhite_missed_idx[0]
    print(f"\nCase 2: Non-White patient MISSED by the model (false negative)")
    print(f"  True label: {y_test[idx]}, Predicted: {y_pred_gb[idx]}")
    shap.force_plot(explainer.expected_value, shap_values[idx], X_test_df.iloc[idx],
                    matplotlib=True, show=False)
    plt.title("Non-White Patient — MISSED by Model (False Negative)", fontsize=12)
    plt.tight_layout()
    plt.savefig('figures/fig5b_shap_force_nonwhite.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n  → This patient needed intensive care but the model failed to flag them.")
    print("    SHAP reveals which features pushed the prediction toward 'low utilizer'.")

## 5.1 Fairness-Through-Blindness Ablation

A common but mistaken intuition is that *removing the protected attribute*
from the model's features is enough to make it fair. Our SHAP analysis
suggested otherwise: features like `Income category` and `Insurance coverage` correlate
with `Race` and can encode it indirectly. We test this empirically by
retraining the same Gradient Boosting model with the `Race` feature dropped,
then re-auditing it against the same protected attribute.


In [ ]:
# Drop the Race feature from training, but keep it as the sensitive attribute
# we audit against.
race_idx = feature_cols.index('Race')
X_train_no_race = np.delete(X_train, race_idx, axis=1)
X_test_no_race  = np.delete(X_test,  race_idx, axis=1)

gb_no_race = GradientBoostingClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1, random_state=RANDOM_STATE
)
gb_no_race.fit(X_train_no_race, y_train)
y_pred_no_race = gb_no_race.predict(X_test_no_race)

acc_nr = accuracy_score(y_test, y_pred_no_race)
dp_nr  = demographic_parity_difference(y_test, y_pred_no_race, sensitive_features=sf_test_labeled)
eo_nr  = equalized_odds_difference  (y_test, y_pred_no_race, sensitive_features=sf_test_labeled)
dpr_nr = demographic_parity_ratio   (y_test, y_pred_no_race, sensitive_features=sf_test_labeled)

print("=" * 70)
print("ABLATION: Gradient Boosting without the Race feature")
print("=" * 70)
print(f"Accuracy:                      {acc_nr:.4f}")
print(f"Demographic Parity Difference: {dp_nr:.4f}")
print(f"Demographic Parity Ratio:      {dpr_nr:.4f}")
print(f"Equalized Odds Difference:     {eo_nr:.4f}")

print("\nInterpretation:")
print("  If DP/EO are close to 0, removing Race is sufficient and proxy")
print("  bias is negligible. If DP/EO remain meaningfully positive, the")
print("  proxy-discrimination thesis is empirically supported: features")
print("  like Income category and Insurance coverage carry racial signal")
print("  even with Race removed.")


## 6. Bias Mitigation (Phase 3)

We apply two algorithmic fairness interventions from **Microsoft Fairlearn**:

1. **ThresholdOptimizer** (Post-processing): Adjusts decision thresholds per demographic group to satisfy fairness constraints *after* the model is trained. This is the least invasive approach.

2. **ExponentiatedGradient** (In-processing): Incorporates fairness constraints *during* model training using a reduction-based approach. This is more principled but modifies the model itself.

We compare both mitigated models against the original model to quantify the **Fairness–Accuracy Tradeoff** — a core concept in algorithmic fairness.

In [ ]:
# Mitigation 1: ThresholdOptimizer (Post-processing)
print("Training ThresholdOptimizer (post-processing mitigation)...")
threshold_optimizer = ThresholdOptimizer(
    estimator=gb_model,
    constraints="equalized_odds",
    prefit=True,
    predict_method='predict_proba',  # use probability scores so TO finds finer per-group thresholds
)
threshold_optimizer.fit(X_train, y_train, sensitive_features=sf_train)
y_pred_to = threshold_optimizer.predict(X_test, sensitive_features=sf_test)
print("ThresholdOptimizer fitted and predictions generated.")

# Mitigation 2: ExponentiatedGradient (In-processing)
# Use the same GB backbone as the baseline so the comparison isolates the
# effect of the fairness constraint, not the choice of base estimator.
print("\nTraining ExponentiatedGradient (in-processing mitigation, GB backbone)...")
mitigator = ExponentiatedGradient(
    estimator=GradientBoostingClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.1, random_state=RANDOM_STATE
    ),
    constraints=EqualizedOdds()
)
mitigator.fit(X_train, y_train, sensitive_features=sf_train)  # GB doesn't need scaling
y_pred_eg = mitigator.predict(X_test)
print("ExponentiatedGradient fitted and predictions generated.")

In [ ]:
# Compare all models: Original vs ThresholdOptimizer vs ExponentiatedGradient
models = {
    'Original (GB)': y_pred_gb,
    'GB (no RACE)': y_pred_no_race,        # Ablation: drop the protected attribute
    'ThresholdOptimizer': y_pred_to,        # Post-processing mitigation (EO-targeted)
    'ExponentiatedGradient': y_pred_eg,     # In-processing mitigation (EO-targeted)
}

results = []
for name, preds in models.items():
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds, zero_division=0),
        'DP Difference': demographic_parity_difference(y_test, preds, sensitive_features=sf_test_labeled),
        'DP Ratio': demographic_parity_ratio(y_test, preds, sensitive_features=sf_test_labeled),
        'EO Difference': equalized_odds_difference(y_test, preds, sensitive_features=sf_test_labeled),
    })

results_df = pd.DataFrame(results).set_index('Model')

print("=" * 75)
print("FAIRNESS–ACCURACY TRADEOFF: Before vs. After Mitigation")
print("=" * 75)
print(results_df.to_string(float_format='{:.4f}'.format))

print("\n\nKey:")
print("  DP Difference → 0 = fair (lower is better)")
print("  DP Ratio → 1.0 = fair (higher is better, threshold: 0.8)")
print("  EO Difference → 0 = fair (lower is better)")

In [ ]:
# Visualization: Fairness-Accuracy Tradeoff
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#F44336', '#9C27B0', '#2196F3', '#4CAF50']
markers = ['o', '^', 's', 'D']

# 1. Accuracy vs Demographic Parity Difference
for i, (name, row) in enumerate(results_df.iterrows()):
    axes[0].scatter(abs(row['DP Difference']), row['Accuracy'], 
                    c=colors[i], marker=markers[i], s=200, zorder=5, edgecolors='black')
    axes[0].annotate(name, (abs(row['DP Difference']), row['Accuracy']),
                     textcoords="offset points", xytext=(10, 10), fontsize=10)
axes[0].set_xlabel('|Demographic Parity Difference| (lower = fairer)')
axes[0].set_ylabel('Accuracy (higher = better)')
axes[0].set_title('Accuracy vs. Demographic Parity')
axes[0].axvline(x=0, color='green', linestyle='--', alpha=0.3, label='Perfect fairness')
axes[0].legend()

# 2. Accuracy vs Equalized Odds Difference
for i, (name, row) in enumerate(results_df.iterrows()):
    axes[1].scatter(abs(row['EO Difference']), row['Accuracy'],
                    c=colors[i], marker=markers[i], s=200, zorder=5, edgecolors='black')
    axes[1].annotate(name, (abs(row['EO Difference']), row['Accuracy']),
                     textcoords="offset points", xytext=(10, 10), fontsize=10)
axes[1].set_xlabel('|Equalized Odds Difference| (lower = fairer)')
axes[1].set_ylabel('Accuracy (higher = better)')
axes[1].set_title('Accuracy vs. Equalized Odds')
axes[1].axvline(x=0, color='green', linestyle='--', alpha=0.3, label='Perfect fairness')
axes[1].legend()

plt.suptitle('Fairness–Accuracy Tradeoff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig6_fairness_accuracy_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Results Summary & Discussion

### Summary of Findings

In [ ]:
# Final summary table
print("=" * 75)
print("COMPLETE AUDIT RESULTS SUMMARY")
print("=" * 75)
print()
print(results_df.to_string(float_format='{:.4f}'.format))
print()

# Compute improvement percentages
orig_dp = abs(results_df.loc['Original (GB)', 'DP Difference'])
orig_eo = abs(results_df.loc['Original (GB)', 'EO Difference'])

for mitigated in ['GB (no RACE)', 'ThresholdOptimizer', 'ExponentiatedGradient']:
    mit_dp = abs(results_df.loc[mitigated, 'DP Difference'])
    mit_eo = abs(results_df.loc[mitigated, 'EO Difference'])
    acc_drop = results_df.loc['Original (GB)', 'Accuracy'] - results_df.loc[mitigated, 'Accuracy']
    
    dp_improvement = ((orig_dp - mit_dp) / orig_dp * 100) if orig_dp > 0 else 0
    eo_improvement = ((orig_eo - mit_eo) / orig_eo * 100) if orig_eo > 0 else 0
    
    print(f"\n{mitigated}:")
    print(f"  DP Improvement: {dp_improvement:+.1f}%")
    print(f"  EO Improvement: {eo_improvement:+.1f}%")
    print(f"  Accuracy Cost:  {acc_drop:+.4f}")

### Discussion

**1. Bias was confirmed — and the model amplifies it.** The original Gradient
Boosting model has a Demographic Parity Difference of **0.137** and an
Equalized Odds Difference of **0.192** between White and Non-White patients.
The DP Ratio of **0.34** is well below the 4/5ths-rule threshold of 0.80,
indicating disparate impact under standard anti-discrimination frameworks.
More importantly, the model does not merely *reflect* the underlying
disparity — it *amplifies* it: the ground-truth high-utilization rate is
24.4% for White patients and 13.1% for Non-White patients (a ratio of
1.86×), but the model predicts high utilization for them at rates of 20.7%
and 7.1% respectively (a ratio of 2.93×) — **a 57% amplification of the
underlying disparity**. This is the empirical version of the
Obermeyer et al. (2019) finding for our model.

**2. SHAP plus an explicit ablation confirm proxy discrimination.** The
SHAP dependence plots showed that features like `Income category` (MEPS code
`POVCAT`) and `Insurance coverage` (MEPS code `INSCOV`) interact with `Race`.
To turn this suggestion into evidence, we retrained the same Gradient
Boosting model with the `Race` feature dropped entirely (the "fairness
through blindness" intervention). The result: DP fell from 0.137 to **0.104** and EO from
0.192 to **0.107** — meaningful reductions, but **not zero**. Removing
the protected attribute alone leaves substantial residual bias, because
socioeconomic features carry racial signal indirectly. Practitioners who
"just drop the race column" should not assume the model is now fair.

**3. Three interventions, and ThresholdOptimizer Pareto-dominates.** The
four-row comparison reveals an empirical surprise: on this dataset, *all
three interventions improve accuracy slightly while improving fairness*,
and ThresholdOptimizer dominates the others on every axis.

- **GB without Race** — drops the protected attribute from features only.
  DP improves by **24%**, EO by **44%**, with a *negative* accuracy cost
  (0.8598 vs. 0.8575). Cheap, but the residual gap is large.
- **ThresholdOptimizer** (post-processing, EO-targeted, configured with
  `predict_method='predict_proba'` so per-group thresholds can sit anywhere
  in [0, 1]) — adjusts decision thresholds per demographic group after
  training. **Pareto-dominates the other interventions:** highest accuracy
  (0.8607, +0.32 pts vs baseline), lowest DP Difference (0.0693, **−49%**),
  lowest EO Difference (0.0293, **−85%**).
- **ExponentiatedGradient** (in-processing, EO-targeted, GB backbone) —
  incorporates the EO constraint during training via a reductions
  approach. A middle ground: DP improves by **37%**, EO by **69%**, at
  essentially zero accuracy cost (0.8582, +0.0007 vs. baseline).

Both Fairlearn mitigators were configured with the **EqualizedOdds**
constraint, so the DP improvements are a side-effect of EO optimisation,
not targeted DP interventions. The ThresholdOptimizer-vs-ExponentiatedGradient
choice is therefore not a fairness-vs-accuracy question; it is a
*deployment-architecture* question (post-processing at inference vs the
fairness constraint baked into a retrained model).

**4. Why we report DP and EO, but lean on EO.** Demographic Parity asks
the model to predict positives at the same rate for every group, regardless
of underlying need. In healthcare, where ground-truth health needs can
legitimately differ across populations, DP can be the wrong target — strict
DP would force under-prediction for the higher-need group. Equalized Odds
is more defensible because it conditions on ground truth (TPR and FPR per
group). We still report DP because the 4/5ths rule operates on it and
because it makes the *amplification* finding above visible. EO is the
primary criterion for choosing between models; DP is a secondary check.

**5. SDAIA Alignment Summary.**

| SDAIA Principle | Status |
|---|---|
| Fairness | Audited (DP, EO, base-rate amplification) and mitigated via Fairlearn (drop-RACE ablation, ThresholdOptimizer, ExponentiatedGradient) |
| Transparency & Explainability | Global + individual explanations via SHAP TreeExplainer; proxy bias hypothesised (SHAP) and confirmed (ablation) |
| Reliability & Safety *(Safety dimension only)* | ThresholdOptimizer reduced FNR disparity from 0.192 (EO) to 0.029, lowering missed-case risk for the disadvantaged group — at no accuracy cost. P5's Reliability dimension — system-as-designed behaviour — is out of scope. |
| Accountability & Responsibility *(human-oversight requirement only)* | SHAP enables case-level review; mitigation choices are documented and reproducible (seed = 42). Broader P7 obligations — governance, liability, lifecycle management — are out of scope. |

### Limitations

- The binary White vs. Non-White grouping collapses very different sub-populations (Hispanic, Black, Asian, Multi-racial). It also obscures that **Non-White is the numerical majority of MEPS Panel 19** (10,174 of 15,830 = 64.3%) — disadvantage here is in selection rate and recall, not population size.
- MEPS utilization data is itself shaped by access barriers. Equalized Odds uses utilization as ground truth; if that ground truth is biased, EO understates the true unfairness. This is the Obermeyer thesis and it bounds what any technical intervention can achieve.
- Mitigation techniques optimise mathematical fairness — equal rates across groups — not substantive justice (which would require equalising health *outcomes*, not just predictions).
- The dataset is from the U.S. context; findings on demographic disparities may not transfer directly to Saudi Arabia, but the methodology and SDAIA principle alignment do.

### Future Work

- Apply the same audit methodology to Saudi healthcare data (e.g., from SDAIA's data catalog) so the SDAIA-aligned methodology is exercised on a domestic dataset.
- Explore intersectional fairness (race × gender × age) — the binary group analysis here is a starting point, not an endpoint.
- Compare additional mitigation strategies (pre-processing reweighing, adversarial debiasing) and target-criterion variants (DP-targeted vs EO-targeted).
- Extend to multi-class utilization levels rather than the ≥10-visit binary cut, so the model preserves more clinical signal.


## 8. Conclusion

This technical AI ethics audit walked through a complete Phase 2 → Phase 3 pipeline for identifying, explaining, and mitigating racial bias in a healthcare machine learning system, on the MEPS Panel 19 dataset (n = 15,830):

1. **Bias detection.** Using Fairlearn's `MetricFrame`, we quantified disparities in Demographic Parity (0.137) and Equalized Odds (0.192) between White and Non-White patients, and showed that the model amplifies the underlying base-rate disparity by 57%.

2. **Bias explanation.** Using SHAP TreeExplainer, we identified both the direct effect of `Race` and the indirect effect of socioeconomic proxies (`Income category`, `Insurance coverage`). An explicit drop-`Race` ablation confirmed that "fairness through blindness" is insufficient: residual DP of 0.104 and EO of 0.107 remain even with the protected attribute removed.

3. **Bias mitigation.** We implemented three interventions and characterised their Fairness–Accuracy profiles. `ThresholdOptimizer` (post-processing, `predict_proba` thresholds) Pareto-dominates the others on this dataset — highest accuracy (0.8607, +0.32 pts), lowest DP (0.0693, −49%), lowest EO (0.0293, −85%). `ExponentiatedGradient` with a GB backbone gives moderate fairness gains (DP −37%, EO −69%) at near-zero accuracy cost. Dropping `Race` alone gives partial fairness gains and confirms the proxy-discrimination thesis.

4. **SDAIA compliance.** The audit methodology engages four of the seven SDAIA AI Ethics Principles (May 2025, SDAIA-P114E): it operationalises *Fairness* (P1) and *Transparency & Explainability* (P6) directly, and addresses the *Safety* dimension of *Reliability & Safety* (P5) and the *human-oversight* requirement of *Accountability & Responsibility* (P7). The remaining halves of P5 and P7, and principles P2–P4, are out of scope.

**Recommendation.** Deploy `ThresholdOptimizer`. It Pareto-dominates the other interventions on this dataset — better accuracy *and* better fairness than the unmitigated baseline, with no model retraining required (only per-group probability thresholds applied at inference). `ExponentiatedGradient` remains a sensible fallback if the deployment context requires the fairness constraint to be enforced inside the trained model. In every deployment, route SHAP-flagged edge cases to human reviewers — as required by the SDAIA *Accountability & Responsibility* principle, which calls for demonstrable human oversight, governance, and management of AI systems.

---

### References

1. Obermeyer, Z., Powers, B., Vogeli, C., & Mullainathan, S. (2019). Dissecting racial bias in an algorithm used to manage the health of populations. *Science*, 366(6464), 447-453.
2. SDAIA (May 2025). *AI Ethics Principles*, Document SDAIA-P114E, Version 1. Saudi Data and Artificial Intelligence Authority. https://sdaia.gov.sa/en/SDAIA/about/Documents/ai-principles.pdf
3. Bird, S., et al. (2020). Fairlearn: A toolkit for assessing and improving fairness in AI. *Microsoft Research*.
4. Lundberg, S. M., & Lee, S. I. (2017). A unified approach to interpreting model predictions. *NeurIPS*.
5. AHRQ. Medical Expenditure Panel Survey. Agency for Healthcare Research and Quality.
